In [3]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI

# ==========================================
# 1. STANDALONE SETUP & GROQ CONNECTION
# ==========================================
load_dotenv()

# Initialize standard OpenAI client for Groq
client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

# Hardcode known stable Groq models with tool calling support
PREFERRED_TOOL_MODELS = [
    "openai/gpt-oss-120b"
]

try:
    available_models = [m.id for m in client.models.list().data]
    # Pick the first supported model actually present in your account's model list
    MODEL = next((m for m in PREFERRED_TOOL_MODELS if m in available_models), "openai/gpt-oss-120b")
except Exception:
    MODEL = "openai/gpt-oss-120b"

print(f"✅ Active Model with Tool Support: {MODEL}")
# ==========================================
# 2. STATE & STORAGE
# ==========================================
BUDGET_DATA = {
    "monthly_budget": 50000.0,
    "savings_goal": 10000.0,
    "category_limits": {
        "Food": 5000.0,
        "Education": 10000.0,
        "Entertainment": 4000.0
    },
    "expenses": []
}

# ==========================================
# 3. CORE AGENT TOOLS
# ==========================================
def add_expense(item: str, amount: float, category: str = "General") -> str:
    """Record an expense in Rs and check category limits."""
    amount = float(amount)
    cat_fmt = category.capitalize()
    
    BUDGET_DATA["expenses"].append({
        "item": item, 
        "amount": amount, 
        "category": cat_fmt
    })
    
    category_total = sum(e["amount"] for e in BUDGET_DATA["expenses"] if e["category"] == cat_fmt)
    category_limit = BUDGET_DATA["category_limits"].get(cat_fmt)
    
    msg = f"Logged expense: '{item}' for Rs {amount:.2f} under [{cat_fmt}]."
    if category_limit and category_total > category_limit:
        over_by = category_total - category_limit
        msg += f" ⚠️ WARNING: Category '{cat_fmt}' is overspent by Rs {over_by:.2f}!"
        
    return msg

def get_summary(category: str = "") -> str:
    """Get balance breakdown and usable budget minus savings goal."""
    expenses = BUDGET_DATA["expenses"]
    if category:
        cat_fmt = category.capitalize()
        expenses = [e for e in expenses if e["category"] == cat_fmt]
        
    total_spent = sum(e["amount"] for e in BUDGET_DATA["expenses"])
    cat_spent = sum(e["amount"] for e in expenses)
    remaining_net = BUDGET_DATA["monthly_budget"] - total_spent
    usable_after_savings = remaining_net - BUDGET_DATA["savings_goal"]
    
    item_list = ", ".join([f"{e['item']} (Rs {e['amount']:.2f})" for e in expenses]) if expenses else "None"
    
    return (
        f"Monthly Budget: Rs {BUDGET_DATA['monthly_budget']:.2f} | "
        f"Total Spent: Rs {total_spent:.2f} | "
        f"Savings Goal Buffer: Rs {BUDGET_DATA['savings_goal']:.2f} | "
        f"Usable Available Balance: Rs {usable_after_savings:.2f} | "
        f"Logged Items: {item_list}"
    )

def set_savings_goal(amount: float) -> str:
    """Set or update locked savings target in Rs."""
    BUDGET_DATA["savings_goal"] = float(amount)
    return f"Savings goal updated to Rs {amount:.2f}."

# Pre-flight Tool Self-Test
print("\n--- Pre-flight Tool Self-Test ---")
print(add_expense("Textbooks", 1500, "Education"))
print(get_summary("Education"))
print("-" * 40)

# Reset state after test so execution trace starts clean
BUDGET_DATA["expenses"].clear()

# Tool Registry
REGISTRY = {
    "add_expense": add_expense,
    "get_summary": get_summary,
    "set_savings_goal": set_savings_goal,
}

# Explicit JSON Tool Schema
TOOL_SCHEMA = [
    {
        "type": "function",
        "function": {
            "name": "add_expense",
            "description": "Log an expense in Indian Rupees (Rs) and trigger overspend warnings if applicable.",
            "parameters": {
                "type": "object",
                "properties": {
                    "item": {"type": "string", "description": "Description of item."},
                    "amount": {"type": "number", "description": "Cost in Rs."},
                    "category": {"type": "string", "description": "Category name (e.g. Food, Education)."}
                },
                "required": ["item", "amount"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_summary",
            "description": "Get spending breakdown and usable available balance. MUST be used before deciding affordability.",
            "parameters": {
                "type": "object",
                "properties": {
                    "category": {"type": "string", "description": "Optional category filter."}
                },
                "required": [],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "set_savings_goal",
            "description": "Set or modify monthly savings target in Rupees.",
            "parameters": {
                "type": "object",
                "properties": {
                    "amount": {"type": "number", "description": "Savings goal in Rs."}
                },
                "required": ["amount"],
            },
        },
    },
]

SYSTEM = (
    "You are a Personal Budget Assistant working in Indian Rupees (Rs). "
    "Always use 'add_expense' when a user mentions spending money. "
    "Always use 'get_summary' to inspect actual usable balance before answering whether a user can afford something. "
    "When you have everything you need, answer the user directly and stop calling tools."
)

# ==========================================
# 4. REASONING LOOP WITH MEMORY
# ==========================================
def run_agent(messages: list, goal: str, max_steps: int = 6, verbose: bool = True) -> str:
    messages.append({"role": "user", "content": goal})

    for step in range(1, max_steps + 1):
        # THINK
        response = client.chat.completions.create(
            model=MODEL, messages=messages, tools=TOOL_SCHEMA
        )
        message = response.choices[0].message

        messages.append(message.model_dump(exclude_none=True))

        # EXIT 1: Model finished decision loop
        if not message.tool_calls:
            if verbose:
                print(f"[step {step}] Final Response Ready.")
            return message.content or ""

        # ACT & OBSERVE
        for call in message.tool_calls:
            name = call.function.name
            args = json.loads(call.function.arguments or "{}")

            if name in REGISTRY:
                result = REGISTRY[name](**args)
            else:
                result = f"Error: no tool named '{name}'."

            if verbose:
                print(f"[step {step}] {name}({args}) -> {result}")

            messages.append(
                {"role": "tool", "tool_call_id": call.id, "content": result}
            )

    return f"Stopped after {max_steps} steps without reaching a final answer."

print("\nAgent initialized and ready for execution trace.\n")

# ==========================================
# 5. MULTI-TURN DEMO EXECUTION TRACE
# ==========================================
history = [{"role": "system", "content": SYSTEM}]

print("=== Turn 1 ===")
ans1 = run_agent(history, "I spent Rs 5500 on Food and Rs 1200 on Books.")
print("\nAnswer 1:\n", ans1)
print("\n" + "="*50 + "\n")

print("=== Turn 2 ===")
ans2 = run_agent(history, "Set my target savings goal to Rs 15000.")
print("\nAnswer 2:\n", ans2)
print("\n" + "="*50 + "\n")

print("=== Turn 3 ===")
ans3 = run_agent(history, "Can I afford to buy a Rs 30000 laptop right now?")
print("\nAnswer 3:\n", ans3)

✅ Active Model with Tool Support: openai/gpt-oss-120b

--- Pre-flight Tool Self-Test ---
Logged expense: 'Textbooks' for Rs 1500.00 under [Education].
Monthly Budget: Rs 50000.00 | Total Spent: Rs 1500.00 | Savings Goal Buffer: Rs 10000.00 | Usable Available Balance: Rs 38500.00 | Logged Items: Textbooks (Rs 1500.00)
----------------------------------------

Agent initialized and ready for execution trace.

=== Turn 1 ===
[step 1] add_expense({'amount': 5500, 'category': 'Food', 'item': 'Food expense'}) -> Logged expense: 'Food expense' for Rs 5500.00 under [Food]. ⚠️ WARNING: Category 'Food' is overspent by Rs 500.00!
[step 2] add_expense({'amount': 1200, 'category': 'Books', 'item': 'Books'}) -> Logged expense: 'Books' for Rs 1200.00 under [Books].
[step 3] Final Response Ready.

Answer 1:
 Your expenses have been recorded:

- Food: Rs 5,500  
- Books: Rs 1,200  

Let me know if you’d like a summary of your spending or need help planning any upcoming purchases!


=== Turn 2 ===
[step

EXPECTED OUTPUT
---------------

✅ Connected to Groq API successfully!
✅ Active Model: llama-3.1-8b-instant

Pre-flight Tool Self-Test
Logged expense: 'Textbooks' for Rs 1500.00 under [Education].
Monthly Budget: Rs 50000.00 | Total Spent: Rs 1500.00 | Savings Goal Buffer: Rs 10000.00 | Usable Available Balance: Rs 38500.00 | Logged Items: Textbooks (Rs 1500.00)

Agent initialized and ready for execution trace.

=== Turn 1 ===
[step 1] add_expense({'item': 'Food', 'amount': 5500, 'category': 'Food'}) -> Logged expense: 'Food' for Rs 5500.00 under [Food]. ⚠️ WARNING: Category 'Food' is overspent by Rs 500.00!
[step 1] add_expense({'item': 'Books', 'amount': 1200, 'category': 'Education'}) -> Logged expense: 'Books' for Rs 1200.00 under [Education].
[step 2] Final Response Ready.

Answer 1:
I have recorded your expenses:
- **Food:** Rs 5,500.00 (⚠️ Category limit exceeded by Rs 500.00)
- **Books:** Rs 1,200.00 under Education

==================================================

=== Turn 2 ===
[step 1] set_savings_goal({'amount': 15000}) -> Savings goal updated to Rs 15000.00.
[step 2] Final Response Ready.

Answer 2:
Your target monthly savings goal has been updated to **Rs 15,000.00**. This balance is now reserved from your spending pool.

==================================================

=== Turn 3 ===
[step 1] get_summary({}) -> Monthly Budget: Rs 50000.00 | Total Spent: Rs 6700.00 | Savings Goal Buffer: Rs 15000.00 | Usable Available Balance: Rs 28300.00 | Logged Items: Food (Rs 5500.00), Books (Rs 1200.00)
[step 2] Final Response Ready.

Answer 3:
No, you cannot afford the Rs 30,000 laptop right now. 
Here is your current financial breakdown:
- **Total Monthly Allowance:** Rs 50,000.00
- **Total Spent So Far:** Rs 6,700.00
- **Reserved Savings Goal:** Rs 15,000.00
- **Usable Balance Left:** Rs 28,300.00

You are short by **Rs 1,700.00**.